# Wav2Vec2 Swahili ASR - Local Evaluation (Mac)
WER & CER Evaluation

In [9]:
import torch
print(torch.__version__)


2.2.2


In [10]:
# Install dependencies (run once)
# !pip install transformers datasets evaluate jiwer torch torchaudio

In [ ]:
import os
import re
import torch
import torchaudio
import pandas as pd
import numpy as np
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import evaluate

# Use CPU - MPS has compatibility issues with wav2vec2
device = torch.device("cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [12]:
# Configuration
repo_name = 'jkhyjkhy/wav2vec2-large-xlsr-sw-ASR'

# Local paths
BASE = '/Users/annette/Desktop/automatic_speech_recognition/project/git_folder/LR_ASR/preprocessing/processed_data/14.0-delta-2023-06-23'
CSV = f'{BASE}/manifest_sw_14_0_delta.csv'
AUDIO_DIR = f'{BASE}/cleaned_sw_audio_14_0_delta'

print(f"Base path: {BASE}")
print(f"CSV path: {CSV}")
print(f"Audio dir: {AUDIO_DIR}")

Base path: /Users/annette/Desktop/automatic_speech_recognition/project/git_folder/LR_ASR/preprocessing/processed_data/14.0-delta-2023-06-23
CSV path: /Users/annette/Desktop/automatic_speech_recognition/project/git_folder/LR_ASR/preprocessing/processed_data/14.0-delta-2023-06-23/manifest_sw_14_0_delta.csv
Audio dir: /Users/annette/Desktop/automatic_speech_recognition/project/git_folder/LR_ASR/preprocessing/processed_data/14.0-delta-2023-06-23/cleaned_sw_audio_14_0_delta


In [13]:
# Load dataset
df = pd.read_csv(CSV)
df["wav_path"] = df["wav_path"].apply(lambda p: os.path.join(AUDIO_DIR, os.path.basename(p)))
print(f"Total samples in dataset: {len(df)}")
df.head()

Total samples in dataset: 270


,wav_path,duration,transcript
0,/Users/annette/Desktop/automatic_speech_recogn...,3.252,Ugongwa wa kupinda shingo
1,/Users/annette/Desktop/automatic_speech_recogn...,5.979,Dalili ya ugonjwa wa miguu na midomo ni mnyama...
2,/Users/annette/Desktop/automatic_speech_recogn...,4.046,Wanyama walioathirika na ugonjwa wa ukurutu wa...
3,/Users/annette/Desktop/automatic_speech_recogn...,7.815,Hata hivyo maelfu ya watu waliojitokeza kumsik...
4,/Users/annette/Desktop/automatic_speech_recogn...,7.174,Marais Uhuru Kenyata wa Kenya Yoweri Museveni ...


In [14]:
# Text preprocessing function
def remove_special_characters(text):
    chars_to_ignore_regex = '[\,\?\.\'\!\-\;\:\"\(\)]'
    return re.sub(chars_to_ignore_regex, '', text).lower() + " "

# Apply preprocessing to dataframe
df["transcript"] = df["transcript"].apply(remove_special_characters)
print(f"Preprocessed {len(df)} samples")

Preprocessed 270 samples


In [15]:
# Load model and processor from HuggingFace
print("Loading model and processor...")
processor = Wav2Vec2Processor.from_pretrained(repo_name)
model = Wav2Vec2ForCTC.from_pretrained(repo_name)
model.to(device)
model.eval()
print("Model loaded successfully!")

Loading model and processor...


/Users/annette/Desktop/automatic_speech_recognition/project/git_folder/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model loaded successfully!


In [16]:
# Prepare dataset with audio features (using torchaudio)
def prepare_dataset_manual(row):
    # Load audio using torchaudio
    waveform, sr = torchaudio.load(row["wav_path"])
    
    # Resample to 16kHz if needed
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)
    
    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    audio_array = waveform.squeeze().numpy()
    
    # Process audio
    audio_inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
    input_values = audio_inputs["input_values"][0]
    
    # Process text
    labels = processor.tokenizer(row["transcript"])["input_ids"]
    
    return {
        "input_values": input_values.numpy().tolist(),
        "labels": labels,
        "transcript": row["transcript"]
    }

print("Preparing dataset...")
processed_data = []
skipped = []

for idx, row in df.iterrows():
    if idx % 50 == 0:
        print(f"Processing {idx}/{len(df)}...")
    try:
        processed_data.append(prepare_dataset_manual(row))
    except Exception as e:
        skipped.append((idx, row["wav_path"], str(e)))
        print(f"  Skipped {idx}: {os.path.basename(row['wav_path'])} - {e}")

print(f"\nDataset ready: {len(processed_data)} samples")
print(f"Skipped: {len(skipped)} files")

Preparing dataset...
Processing 0/270...
  Skipped 2: common_voice_sw_37772967.wav - Error loading audio file: failed to open file /Users/annette/Desktop/automatic_speech_recognition/project/git_folder/LR_ASR/preprocessing/processed_data/14.0-delta-2023-06-23/cleaned_sw_audio_14_0_delta/common_voice_sw_37772967.wav
  Skipped 5: common_voice_sw_37990259.wav - Error loading audio file: failed to open file /Users/annette/Desktop/automatic_speech_recognition/project/git_folder/LR_ASR/preprocessing/processed_data/14.0-delta-2023-06-23/cleaned_sw_audio_14_0_delta/common_voice_sw_37990259.wav
  Skipped 6: common_voice_sw_37803619.wav - Error loading audio file: failed to open file /Users/annette/Desktop/automatic_speech_recognition/project/git_folder/LR_ASR/preprocessing/processed_data/14.0-delta-2023-06-23/cleaned_sw_audio_14_0_delta/common_voice_sw_37803619.wav
  Skipped 9: common_voice_sw_37840765.wav - Error loading audio file: failed to open file /Users/annette/Desktop/automatic_speech_r

In [17]:
# Load evaluation metrics
wer_metric = evaluate.load('wer')
cer_metric = evaluate.load('cer')
print("Metrics loaded: WER, CER")

Metrics loaded: WER, CER


In [18]:
# Run inference
print("Running inference...")
predictions = []
references = []

for idx, sample in enumerate(processed_data):
    if idx % 50 == 0:
        print(f"Inferencing {idx}/{len(processed_data)}...")
    
    with torch.no_grad():
        input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)
        logits = model(input_values).logits
    
    pred_ids = torch.argmax(logits, dim=-1)
    pred_str = processor.batch_decode(pred_ids)[0]
    ref_str = processor.decode(sample["labels"], group_tokens=False)
    
    predictions.append(pred_str)
    references.append(ref_str)

print(f"Inference complete: {len(predictions)} samples")

Running inference...
Inferencing 0/209...
Inferencing 50/209...
Inferencing 100/209...
Inferencing 150/209...
Inferencing 200/209...
Inference complete: 209 samples


In [19]:
# Calculate WER and CER
wer_score = wer_metric.compute(predictions=predictions, references=references)
cer_score = cer_metric.compute(predictions=predictions, references=references)

print("="*50)
print("Evaluation Results")
print("="*50)
print(f"Test WER: {wer_score:.4f} ({wer_score*100:.2f}%)")
print(f"Test CER: {cer_score:.4f} ({cer_score*100:.2f}%)")
print("="*50)

Evaluation Results
Test WER: 1.0000 (100.00%)
Test CER: 0.9998 (99.98%)


In [20]:
# Show sample results
from IPython.display import display, HTML
import random

results_df = pd.DataFrame({
    "reference": references,
    "prediction": predictions
})

# Show random samples
sample_indices = random.sample(range(len(results_df)), min(10, len(results_df)))
display(HTML(results_df.iloc[sample_indices].to_html()))

,reference,prediction
193,kukosa utulivu kushindwa kusimama imara ulimi kutoka nje kuinua miguu sana wakati wa kutembea,
113,jamii maskini nje ya mji wa juba mjini,
163,ongeza maji ya hamira uliyochachusha kwenye unga,
203,magari ya kubeba wanyama yanaweza kueneza ugonjwa wa miguu na midomo kwa ngombe,
13,ramani mpya ya jumuiya ya afrika mashariki yazinduliwa,
143,takwimu za januari ziko juu kuliko pato la mauzo ya nje kutoka kenya,
94,ukosefu wa vitamini a iliyo kwenye viazi lishe hupelekea mtoto kuugua surua,
133,mjumbe mpya wa jamuhuri ya afrika ya kati anatoa wito wa kurekebishwa kikosi cha kulinda amani cha umoja wa mataifa,
189,kumi na nne hadi shilingi elfu mbili mia nane sitini na moja za tanzania dola moja,
145,tumia kijiko kimoja cha unga ngano,


In [21]:
# Save results to CSV
results_df.to_csv(f"{BASE}/evaluation_results.csv", index=False)
print(f"Results saved to {BASE}/evaluation_results.csv")

Results saved to /Users/annette/Desktop/automatic_speech_recognition/project/git_folder/LR_ASR/preprocessing/processed_data/14.0-delta-2023-06-23/evaluation_results.csv
